In [387]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/store_data.csv")
df.head()
df=df.copy()

In [388]:
df[df["Postal Code"].isna()][["City", "Postal Code"]]

,City,Postal Code
53,New York City,NaN
75,Houston,NaN
93,Minneapolis,NaN
118,Bristol,NaN
141,San Francisco,NaN
...,...,...
9876,Cleveland,NaN
9888,Utica,NaN
9942,Anaheim,NaN
9964,Newark,NaN


In [389]:

df["City"] = df["City"].str.strip().str.lower()
# df.groupby("City")["Postal Code"].describe()
# counts = df.groupby("City")["Postal Code"].nunique()
# counts[counts == 0]
df[df["Postal Code"].isna()][["City", "Postal Code"]]
def postal_replace(city):
    most_postal = df[(df["City"]==city)].groupby("Postal Code")["City"].agg(lambda x: x.mode()[0])
    # df[df["City"] == city].groupby("Postal Code")["City"].value_counts()
    df.loc[df["City"] == city, "Postal Code"] = most_postal.index[0]

cites = df["City"].unique()
for city in cites:
    postal_replace(city)



In [390]:
df[df["Quantity"].isna()][["Quantity", "Sales", "Ship Date", "Product Name", "Ship Mode", "Order ID"]]
df = df.dropna(subset=['Quantity', 'Sales'], how ='all')
df[df["Quantity"].isna()][["Quantity", "Sales", "Ship Date", "Product Name", "Ship Mode", "Order ID"]]


,Quantity,Sales,Ship Date,Product Name,Ship Mode,Order ID
157,NaN,457.568,NaN,Global Deluxe High-Back Manager's Chair,Second Class,CA-2014-104269
310,NaN,466.768,9/14/2016,Hon 4070 Series Pagoda Armless Upholstered Sta...,Second Class,CA-2016-142902
374,NaN,9.240,7/27/2014,Newell 324,Standard Class,US-2014-119137
382,NaN,49.536,10/29/2016,"Square Ring Data Binders, Rigid 75 Pt. Covers,...",First Class,CA-2016-134775
576,NaN,19.920,9/22/2015,Rediform S.O.S. Phone Message Books,Second Class,CA-2015-149713
...,...,...,...,...,...,...
9755,NaN,332.940,3/29/2017,"Carina Mini System Audio Rack, Model AR050B",Second Class,CA-2017-113705
9789,NaN,39.992,4/1/2017,Kensington SlimBlade Notebook Wireless Mouse w...,Standard Class,CA-2017-144491
9862,NaN,12.120,1/20/2017,Imation USB 2.0 Swivel Flash Drive USB flash d...,Standard Class,CA-2017-113278
9914,NaN,12.960,1/31/2017,Xerox 1997,Second Class,CA-2017-160927


In [391]:
df[df["Quantity"].isna() & df["Sales"].isna()][["Quantity", "Sales", "Ship Date", "Product Name", "Ship Mode", "Order ID"]]
df = df.dropna(subset=["Quantity", "Sales"], how="all")


In [392]:
df["Product_general_price"] = (df["Sales"]/df["Quantity"])*(1-df["Discount"])
df["Quantity"] = (df["Sales"]/df["Product_general_price"])


In [393]:
df["Product_general_price"] = (df["Sales"]/df["Quantity"])*(1-df["Discount"])
outliers = df.groupby("Product ID")["Product_general_price"].agg(
    Q1 = lambda sale: sale.quantile(0.25),
    Q3 = lambda sale: sale.quantile(0.75),
    Moyen="mean"
)
outliers["IQR"] = outliers["Q3"] - outliers["Q1"]
outliers["borne_inf"] = (outliers["Q1"] - 1.5 * outliers["IQR"])
outliers["borne_sup"] = (outliers["Q3"] + 1.5 * outliers["IQR"])
df["is_outlier"] = ((df["Product_general_price"] < df["Product ID"].map(outliers["borne_inf"]))|(df["Product_general_price"] > df["Product ID"].map(outliers["borne_sup"])))
product_mean = (df.loc[~df["is_outlier"]].groupby("Product ID")["Product_general_price"].mean())
df["Product_median_price"] = df["Product ID"].map(product_mean)
df.loc[df["Quantity"].isna(), "Quantity"] = (df.loc[df["Quantity"].isna(), "Sales"] * (1 - df.loc[df["Quantity"].isna(), "Discount"]) / df.loc[df["Quantity"].isna(), "Product_median_price"])
df.loc[df["Quantity"]<0, "Quantity"] = (df.loc[df["Quantity"]<0, "Sales"] * (1 - df.loc[df["Quantity"]<0, "Discount"]) / df.loc[df["Quantity"]<0, "Product_median_price"])
df = df.dropna(subset=["Product_median_price"])
df[df["Quantity"].isna()]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Product_general_price,is_outlier,Product_median_price
96,97,CA-2017-161018,11/9/2017,11/11/2017,Second Class,PN-18775,NaN,Home Office,United States,new york city,...,Furniture,Furnishings,9-3/4 Diameter Round Wall Clock,NaN,NaN,0.0,40.5426,NaN,False,11.546827
165,166,CA-2014-139892,9/8/2014,9/12/2014,Standard Class,BM-11140,Becky Martin,Consumer,United States,san antonio,...,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,NaN,NaN,0.4,-1359.9920,NaN,False,550.796760
225,226,CA-2015-163055,8/9/2015,8/16/2015,Standard Class,DS-13180,David Smith,Corporate,United States,detroit,...,Office Supplies,Art,"Sanford Uni-Blazer View Highlighters, Chisel T...",NaN,NaN,0.0,0.9680,NaN,False,2.200000
285,286,CA-2015-130883,9/26/2015,10/2/2015,Standard Class,TB-21520,Tracy Blumstein,Consumerr,United States,portland,...,Office Supplies,Paper,Xerox 216,NaN,NaN,0.2,10.8864,NaN,False,4.898880
413,414,CA-2017-117457,12/8/2017,12/12/2017,Standard Class,KH-16510,Keith Herrera,Consumerr,United States,san francisco,...,Furniture,Chairs,Novimex High-Tech Fabric Mesh Task Chair,NaN,NaN,0.2,-18.4548,NaN,False,47.887840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9775,9776,CA-2014-169019,7/26/2014,7/30/2014,Standard Class,LF-17185,Luke Foster,Consumer,United States,san antonio,...,Furniture,Furnishings,DAX Clear Channel Poster Frame,NaN,NaN,0.6,-10.0602,NaN,False,14.580000
9776,9777,CA-2014-169019,7/26/2014,7/30/2014,Standard Class,LF-17185,Luke Foster,Consumer,United States,san antonio,...,Office Supplies,Binders,GBC Premium Transparent Covers with Diagonal L...,NaN,NaN,0.8,-26.8544,NaN,False,7.549572
9873,9874,CA-2016-100587,12/10/2016,12/14/2016,Standard Class,SL-20155,NaN,Home Office,United States,new york city,...,Office Supplies,Paper,Xerox 221,NaN,NaN,0.0,3.1104,NaN,False,4.371840
9995,1442,CA-2017-128160,12/19/2017,12/24/2017,Second Class,MM-17920,NaN,Consumer,United States,san francisco,...,Office Supplies,Binders,Deluxe Heavy-Duty Vinyl Round Ring Binder,NaN,NaN,0.2,11.4600,NaN,False,9.821220


In [394]:
df.isna().sum()
df.drop(df[df["Quantity"] < 0].index, inplace=True)
df[df["Quantity"]<0]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Product_general_price,is_outlier,Product_median_price


In [395]:
df.isna().sum()

Row ID                     0
Order ID                   0
Order Date                 0
Ship Date                101
Ship Mode                300
Customer ID                0
Customer Name            350
Segment                    0
Country                    0
City                       0
State                      0
Postal Code                0
Region                     0
Product ID                 0
Category                   0
Sub-Category               0
Product Name               0
Sales                    193
Quantity                 193
Discount                   0
Profit                     0
Product_general_price    385
is_outlier                 0
Product_median_price       0
dtype: int64

In [396]:
df["Product Name"] = df["Product Name"].str.title()
p = df.groupby("Product Name")["Product ID"].nunique()
p[p>1]

name_id_count = df.groupby(["Product Name", "Product ID"]).size().reset_index(name="Count")

name_id_count

product_names = df.groupby("Product Name")["Product ID"].nunique()
product_names = product_names[product_names > 1].index
product_names
result = name_id_count[name_id_count["Product Name"].isin(product_names)]
result

mode_id = df.groupby("Product Name")["Product ID"].agg(lambda x: x.mode()[0])
df["Product ID"] = df["Product Name"].map(mode_id)


In [397]:
df[["Order Date","Ship Date", "Ship Mode"]].head(1)
# df[df["Ship Mode"]=="Same Day"][["Order Date","Ship Date"]]

,Order Date,Ship Date,Ship Mode
0,11/8/2016,11/11/2016,Second Class


In [398]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="mixed")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="mixed")
df["Shipping_Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df[df["Ship Date"].isna()][["Order Date","Ship Date", "Ship Mode", "Shipping_Days"]]
df.drop(df[df["Shipping_Days"]<0].index, inplace=True)

In [399]:
shipping_statiss = df.groupby("Ship Mode")["Shipping_Days"].agg(
    Median="median",
    Mean="mean",
    Min="min",
    Max="max",
    Count="count"
)
df[["Order Date","Ship Date", "Ship Mode", "Shipping_Days"]]
df.drop(df[df["Shipping_Days"] < 0].index, inplace=True)
df[df["Shipping_Days"]<0]
# print(shipping_statiss)
df.groupby("Shipping_Days")["Ship Mode"].value_counts()
pd.crosstab(df["Shipping_Days"],df["Ship Mode"])
grouped = df.groupby("Shipping_Days")["Ship Mode"].agg(
    frequency = lambda x: x.mode()[0]
)
grouped.squeeze()
df["Ship Mode"] = df["Ship Mode"].fillna(df["Shipping_Days"].map(grouped.squeeze()))
df[["Order Date","Ship Date", "Ship Mode", "Shipping_Days"]]


,Order Date,Ship Date,Ship Mode,Shipping_Days
0,2016-11-08,2016-11-11,Second Class,3.0
1,2016-11-08,2016-11-11,Second Class,3.0
2,2016-06-12,2016-06-16,Second Class,4.0
3,2015-10-11,2015-10-18,Standard Class,7.0
4,2015-10-11,2015-10-18,Standard Class,7.0
...,...,...,...,...
10059,2016-09-05,2016-09-06,First Class,1.0
10060,2017-12-14,2017-12-18,Standard Class,4.0
10061,2014-05-05,2014-05-07,First Class,2.0
10062,2014-03-07,2014-03-08,First Class,1.0


In [400]:
df.isna().sum()
df[df["Ship Mode"].isna()][["Order Date","Ship Date", "Ship Mode", "Shipping_Days"]]
df.drop(df[df["Ship Mode"].isna()].index, inplace=True)
df.isna().sum()

Row ID                     0
Order ID                   0
Order Date                 0
Ship Date                 99
Ship Mode                  0
Customer ID                0
Customer Name            347
Segment                    0
Country                    0
City                       0
State                      0
Postal Code                0
Region                     0
Product ID                 0
Category                   0
Sub-Category               0
Product Name               0
Sales                    193
Quantity                 193
Discount                   0
Profit                     0
Product_general_price    385
is_outlier                 0
Product_median_price       0
Shipping_Days             99
dtype: int64

In [401]:
# df[df["Ship Date"].isna()][["Order Date","Ship Date", "Ship Mode"]]


df.drop(df[df["Ship Mode"].isna()].index, inplace=True)

shipping_statiss = df.groupby("Ship Mode")["Shipping_Days"].agg(
    Median="median",
    Mean="mean",
    Min="min",
    Max="max",
    Count="count"
)
df[["Order Date","Ship Date", "Ship Mode", "Shipping_Days"]]
df.drop(df[df["Shipping_Days"] < 0].index, inplace=True)
df[df["Shipping_Days"]<0]
# print(shipping_statiss)
df.groupby("Shipping_Days")["Ship Mode"].value_counts()
pd.crosstab(df["Shipping_Days"],df["Ship Mode"])
grouped = df.groupby("Ship Mode")["Shipping_Days"].agg(
    frequency = lambda x: x.mode()[0]
)
grouped
df["Shipping_Days"] = df["Shipping_Days"].fillna(df["Ship Mode"].map(grouped.squeeze()))
df[df["Shipping_Days"].isna()]

df["Order Date"] = pd.to_datetime(df["Order Date"], format="mixed")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="mixed")
df.loc[df["Ship Date"].isna(), "Ship Date"] = df.loc[df["Ship Date"].isna(),"Order Date"] + pd.to_timedelta(df.loc[df["Ship Date"].isna(),"Shipping_Days"], unit="D")
df[df["Ship Date"].isna()][["Order Date","Ship Date", "Ship Mode", "Shipping_Days"]]



,Order Date,Ship Date,Ship Mode,Shipping_Days


In [402]:
df.isna().sum()
df[df["Customer Name"].isna()][["Product Name", "Order ID"]]
cus = df.groupby("Order ID")["Customer Name"].agg(
    freq = lambda x: x.mode()
)

miss = df.groupby("Order ID")["Customer Name"].apply(lambda x: x.isna().all())
misss = miss[miss==True].index
df.drop(df[df["Order ID"].isin(misss)].index, inplace=True)
# data frame indexes
miss[miss==True]
cus
df["Customer Name"] = df["Customer Name"].fillna(df["Order ID"].map(cus.squeeze()))
df[df["Customer Name"].isna()][["Product Name", "Order ID"]]


,Product Name,Order ID


In [403]:
df.isna().sum()
df[df["Sales"].isna()& df["Quantity"].isna()][["Quantity", "Sales", "Ship Date", "Product Name", "Ship Mode", "Order ID"]]
df = df.dropna(subset=['Quantity', 'Sales'], how ='all')
df[df["Sales"].isna()][["Quantity", "Sales", "Ship Date", "Product Name", "Ship Mode", "Order ID"]]

df["Product_general_price"] = (df["Sales"]/df["Quantity"])*(1-df["Discount"])

outliers = df.groupby("Product ID")["Product_general_price"].agg(
    Q1 = lambda sale: sale.quantile(0.25),
    Q3 = lambda sale: sale.quantile(0.75),
    Moyen="mean"
)
outliers
outliers["IQR"] = outliers["Q3"] - outliers["Q1"]
outliers["borne_inf"] = (outliers["Q1"] - 1.5 * outliers["IQR"])
outliers["borne_sup"] = (outliers["Q3"] + 1.5 * outliers["IQR"])
df["is_outlier"] = ((df["Product_general_price"] < df["Product ID"].map(outliers["borne_inf"]))|(df["Product_general_price"] > df["Product ID"].map(outliers["borne_sup"])))
product_mean = (df.loc[~df["is_outlier"]].groupby("Product ID")["Product_general_price"].mean())
product_mean
df["Product_median_price"] = df["Product ID"].map(product_mean)
df.loc[df["Sales"].isna(), "Sales"] = (df.loc[df["Sales"].isna(), "Quantity"] * (1 - df.loc[df["Sales"].isna(), "Discount"]) * df.loc[df["Sales"].isna(), "Product_median_price"])
df = df.dropna(subset=["Product_median_price"])
df[df["Sales"].isna()]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Product_general_price,is_outlier,Product_median_price,Shipping_Days


In [404]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [405]:

df["Segment"].unique()

df["Segment"] = df["Segment"].replace({
    "Consumerr": "Consumer",
    "Corporrate": "Corporate",
    "Home Ofice": "Home Office"
})
df["Segment"].unique()

<StringArray>
['Consumer', 'Corporate', 'Home Office']
Length: 3, dtype: str

In [406]:
df["Category"] = df["Category"].str.title()
df["State"] = df["State"].str.title()
df["Category"].unique()
# df["State"].nunique()

<StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str

In [407]:
df.loc[df["is_outlier"]==True, "Sales"] = (df.loc[df["is_outlier"]==True, "Quantity"] * (1 - df.loc[df["is_outlier"]==True, "Discount"]) * df.loc[df["Sales"].isna(), "Product_median_price"])
df[df["is_outlier"]==True]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Product_general_price,is_outlier,Product_median_price,Shipping_Days
5,6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,los angeles,...,Furnishings,Eldon Expressions Wood And Plastic Desk Access...,NaN,7.000000,0.0,14.1694,6.98000,True,6.980000,5.0
21,22,CA-2016-137330,2016-12-09,2016-12-13,Standard Class,KB-16585,Ken Black,Corporate,United States,fremont,...,Art,Newell 318,NaN,7.000000,0.0,5.0596,2.78000,True,2.780000,4.0
23,24,US-2017-156909,2017-07-16,2017-07-18,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,philadelphia,...,Chairs,"Global Deluxe Stacking Chair, Gray",NaN,2.857143,0.3,-1.0196,17.48614,True,26.101760,2.0
36,37,CA-2016-117590,2016-12-08,2016-12-10,First Class,GH-14485,Gene Hale,Corporate,United States,richardson,...,Furnishings,"Electrix Architect'S Clamp-On Swing Arm Lamp, ...",NaN,12.500000,0.6,-147.9630,6.10944,True,95.460000,2.0
43,44,CA-2017-139619,2017-09-19,2017-09-23,Standard Class,ES-14080,Erin Smith,Corporate,United States,melbourne,...,Storage,"Advantus 10-Drawer Portable Organizer, Chrome ...",NaN,2.500000,0.2,9.5616,30.59712,True,59.760000,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9940,9941,CA-2016-169824,2016-12-12,2016-12-17,Standard Class,NS-18640,Noel Staavos,Corporate,United States,new york city,...,Art,Sanford Pocket Accent Highlighters,NaN,7.000000,0.0,4.8160,1.60000,True,1.600000,5.0
9955,9956,CA-2015-141593,2015-12-14,2015-12-16,Second Class,DB-12970,Darren Budd,Corporate,United States,los angeles,...,Tables,"Bush Andora Conference Table, Maple/Graphite G...",NaN,2.500000,0.2,10.2588,87.54176,True,51.407987,2.0
9961,9962,CA-2015-168088,2015-03-19,2015-03-22,First Class,CM-12655,Corinna Mitchell,Home Office,United States,houston,...,Paper,Xerox 1919,NaN,2.500000,0.2,23.7742,20.98688,True,40.990000,3.0
9969,9970,CA-2017-153871,2017-12-11,2017-12-17,Standard Class,RB-19435,Richard Bierner,Consumer,United States,plainfield,...,Appliances,"Bravo Ii Megaboss 12-Amp Hard Body Upright, Re...",NaN,7.000000,0.0,6.5975,3.25000,True,1.664000,6.0


In [408]:
df["profit_margin"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"], np.nan)

In [411]:
df.shape


(9706, 26)